In [1]:
from pathlib import Path
import pandas as pd
import fitz

PROJECT_ROOT = Path(r"C:\Users\yzhoo\AI-Agent-Job-Search")

RESUME_ROOT = PROJECT_ROOT / "experiments" / "dataset" / "resume dataset"
PDF_ROOT = RESUME_ROOT / "data"

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "dataset" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert PDF_ROOT.exists(), f"Dataset folder not found: {PDF_ROOT}"

print("Resume folder:", RESUME_ROOT)
print("PDF folder:", PDF_ROOT)
print("Output folder:", OUTPUT_DIR)

Resume folder: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\resume dataset
PDF folder: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\resume dataset\data
Output folder: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed


In [2]:
pdf_files = sorted(PDF_ROOT.rglob("*.pdf"))
csv_files = sorted(RESUME_ROOT.rglob("*.csv"))

print("Total PDF resumes:", len(pdf_files))
print("Total CSV files:", len(csv_files))

category_summary = pd.DataFrame([
    {
        "category": folder.name,
        "resume_count": len(list(folder.glob("*.pdf")))
    }
    for folder in sorted(PDF_ROOT.iterdir())
    if folder.is_dir()
])

display(category_summary)

if csv_files:
    print("CSV file:", csv_files[0])
else:
    print("No CSV file found.")

Total PDF resumes: 2484
Total CSV files: 1


,category,resume_count
0,ACCOUNTANT,118
1,ADVOCATE,118
2,AGRICULTURE,63
3,APPAREL,97
4,ARTS,103
5,AUTOMOBILE,36
6,AVIATION,117
7,BANKING,115
8,BPO,22
9,BUSINESS-DEVELOPMENT,120


CSV file: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\resume dataset\Resume\Resume.csv


In [3]:
def read_csv_fallback(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not decode CSV: {path}")


if csv_files:
    csv_df = read_csv_fallback(csv_files[0])

    print("CSV shape:", csv_df.shape)
    print("Columns:", csv_df.columns.tolist())
    display(csv_df.head())
else:
    csv_df = None

CSV shape: (2484, 4)
Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
records = []
errors = []

for i, pdf_path in enumerate(pdf_files, start=1):
    category = pdf_path.parent.name

    try:
        with fitz.open(pdf_path) as document:
            text = "\n".join(
                page.get_text("text")
                for page in document
            ).strip()

        records.append({
            "resume_id": f"R{i:04d}",
            "category": category,
            "file_name": pdf_path.name,
            "relative_path": str(pdf_path.relative_to(PROJECT_ROOT)),
            "text": text,
            "text_char_count": len(text),
            "extraction_status": "ok" if text else "empty_text"
        })

    except Exception as exc:
        records.append({
            "resume_id": f"R{i:04d}",
            "category": category,
            "file_name": pdf_path.name,
            "relative_path": str(pdf_path.relative_to(PROJECT_ROOT)),
            "text": "",
            "text_char_count": 0,
            "extraction_status": "error"
        })

        errors.append({
            "file_path": str(pdf_path),
            "error": str(exc)
        })

    if i % 250 == 0 or i == len(pdf_files):
        print(f"Processed {i}/{len(pdf_files)} PDFs")

resumes_df = pd.DataFrame(records)
errors_df = pd.DataFrame(errors)

print("Total extracted records:", len(resumes_df))
print("Non-empty resumes:", (resumes_df["text_char_count"] > 0).sum())
print("Empty resumes:", (resumes_df["text_char_count"] == 0).sum())
print("Extraction errors:", len(errors_df))

display(
    resumes_df[
        ["resume_id", "category", "file_name",
         "text_char_count", "extraction_status"]
    ].head()
)

Processed 250/2484 PDFs
Processed 500/2484 PDFs
Processed 750/2484 PDFs
Processed 1000/2484 PDFs
Processed 1250/2484 PDFs
Processed 1500/2484 PDFs
Processed 1750/2484 PDFs
Processed 2000/2484 PDFs
Processed 2250/2484 PDFs
Processed 2484/2484 PDFs
Total extracted records: 2484
Non-empty resumes: 2483
Empty resumes: 1
Extraction errors: 0


,resume_id,category,file_name,text_char_count,extraction_status
0,R0001,ACCOUNTANT,10554236.pdf,24162,ok
1,R0002,ACCOUNTANT,10674770.pdf,7493,ok
2,R0003,ACCOUNTANT,11163645.pdf,4749,ok
3,R0004,ACCOUNTANT,11759079.pdf,5922,ok
4,R0005,ACCOUNTANT,12065211.pdf,5574,ok


In [5]:
review_df = resumes_df[
    (resumes_df["text_char_count"] < 100) |
    (resumes_df["extraction_status"] != "ok")
].copy()

print("Resumes needing review:", len(review_df))
display(review_df.head(20))

nonempty = resumes_df[resumes_df["text"].str.strip().ne("")]

if not nonempty.empty:
    sample = nonempty.iloc[0]

    print("Sample category:", sample["category"])
    print("Sample file:", sample["file_name"])
    print("\nExtracted text preview:\n")
    print(sample["text"][:2000])
else:
    print("No selectable text found. OCR may be required.")

Resumes needing review: 1


,resume_id,category,file_name,relative_path,text,text_char_count,extraction_status
802,R0803,BUSINESS-DEVELOPMENT,12632728.pdf,experiments\dataset\resume dataset\data\BUSINE...,,0,empty_text


Sample category: ACCOUNTANT
Sample file: 10554236.pdf

Extracted text preview:

ACCOUNTANT
Summary
Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.
Highlights
Account reconciliations
Results-oriented
Financial reporting
Critical thinking
Accounting operations professional
Analysis of financial systems
ERP (Enterprise Resource Planning) software.
Excellent facilitator
Accomplishments
Served on a tiger team which identified and resolved General Ledger postings in DEAMS totaling $360B in accounting adjustments. This allowed
for the first successful fiscal year-end close for 2012.
In collaboration with DFAS Europe, developed an automated tool that identified duplicate obligations. This tool allowed HQ USAFE to
deobligate over $5M in duplicate obligations.
Experience
Company Name July 2011 to November 2012 Accountant 
City , State
Enterprise Resource Planning Office (ERO)
In this position as an Accountant assigned to the Defen

In [7]:
resumes_df.to_csv(
    OUTPUT_DIR / "resumes_extracted.csv",
    index=False,
    encoding="utf-8-sig"
)

category_summary.to_csv(
    OUTPUT_DIR / "resume_category_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

review_df.to_csv(
    OUTPUT_DIR / "resume_extraction_review.csv",
    index=False,
    encoding="utf-8-sig"
)

errors_df.to_csv(
    OUTPUT_DIR / "resume_extraction_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved processed files to:", OUTPUT_DIR)

Saved processed files to: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed


In [6]:
print("===== RESUME DATASET SUMMARY =====")
print("PDF files discovered:", len(pdf_files))
print("Processed resume records:", len(resumes_df))
print("Unique categories:", resumes_df["category"].nunique())
print("Non-empty text:", (resumes_df["text_char_count"] > 0).sum())
print("Empty text:", (resumes_df["text_char_count"] == 0).sum())
print("Extraction errors:", len(errors_df))

display(
    resumes_df.groupby("category")
    .size()
    .reset_index(name="resume_count")
    .sort_values("resume_count", ascending=False)
)

===== RESUME DATASET SUMMARY =====
PDF files discovered: 2484
Processed resume records: 2484
Unique categories: 24
Non-empty text: 2483
Empty text: 1
Extraction errors: 0


,category,resume_count
20,INFORMATION-TECHNOLOGY,120
9,BUSINESS-DEVELOPMENT,120
0,ACCOUNTANT,118
1,ADVOCATE,118
10,CHEF,118
15,ENGINEERING,118
16,FINANCE,118
6,AVIATION,117
17,FITNESS,117
22,SALES,116
